# 02 平均分散とポートフォリオ最適化

[研究案内](../資料/07_研究コードと成果物の案内.md) / [実装元](../src/quant_ai/portfolio.py)。VS Codeで関数名をF12、または下のセルで実装を表示。

## 1. 目的と仮説
標本内フロンティアと学習外の運用成績を分ける。

## 2. 理論と数式
$\min w'\Sigma w,\ 1'w=1,w\ge0$。接点は$(w'\mu-r_f)/\sqrt{w'\Sigma w}$最大化。

## 3. データの定義
seed42、700営業日×5合成資産。前半60%だけからμ・Σを推定する。

## 4. 分析条件・パラメータ
算術年率平均252×日次平均、共分散252×日次共分散。月次近似21営業日、片道5bps。

## 5. 実装と実行
共通関数の実装も確認してから計算する。

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "src/quant_ai").exists())
os.chdir(ROOT)
(ROOT / "分析結果/図").mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
print("データ種別: 合成データ。実市場での収益性を示す結果ではありません。")


In [ ]:
import inspect
from IPython.display import Code
from quant_ai.portfolio import optimize_portfolio
print(inspect.getsourcefile(optimize_portfolio))
display(Code(inspect.getsource(optimize_portfolio), language='python'))

In [ ]:
from quant_ai.data_loader import synthetic_prices
from quant_ai.returns import simple_returns
from quant_ai.portfolio import optimize_portfolio, efficient_frontier, cash_allocation
from quant_ai.backtest import backtest
from quant_ai.metrics import performance
prices = synthetic_prices(700, 5)
cut = int(len(prices)*0.6)
train = simple_returns(prices.iloc[:cut])
mu, cov = train.mean()*252, train.cov()*252
w = optimize_portfolio(mu, cov)
rf_arithmetic = 252*(1.02**(1/252)-1)
tangent = optimize_portfolio(mu, cov, objective="max_sharpe", risk_free=rf_arithmetic)
display(pd.DataFrame({"min_variance": w, "tangent": tangent}))
display(cash_allocation(tangent, 0.7))
frontier = efficient_frontier(mu, cov)


## 6. 計算結果と可視化

In [ ]:
fig, ax = plt.subplots()
ax.plot(frontier.volatility, frontier["return"], "o-")
ax.scatter(np.sqrt(np.diag(cov)), mu, marker="x", label="individual assets")
ax.set(xlabel="Annual volatility", ylabel="Annual arithmetic expected return",
       title="In-sample efficient frontier - synthetic")
ax.legend()
fig.savefig("分析結果/図/02_効率的フロンティア.png", dpi=150, bbox_inches="tight")
plt.show()
test = prices.iloc[cut:]
signals = pd.DataFrame(np.tile(w.to_numpy(), (len(test), 1)), index=test.index, columns=test.columns)
report = backtest(test, signals, rebalance_every=21, cost_bps=5, benchmark=test.iloc[:, 0])
equal = backtest(test, signals*0+1/prices.shape[1], rebalance_every=21, cost_bps=5)
display(pd.DataFrame({"min_variance": performance(report.net_return),
                      "equal_weight": performance(equal.net_return),
                      "benchmark_gross": performance(report.benchmark_return)}))
ax = pd.DataFrame({"min_variance": report.wealth, "equal_weight": equal.wealth,
                   "benchmark_gross": report.benchmark_wealth}).plot(title="Held-out synthetic period")
ax.figure.savefig("分析結果/図/02_学習外の運用成績.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. 統計的解釈
μ・Σは推定値であり、フロンティアは条件付き。学習外の等金額配分と比較する。

## 8. 金融工学・運用上の解釈
接点・最小分散は目的が違う。無リスク金利の算術/幾何単位も揃える。

## 9. 限界と追加検証
## 解釈
フロンティアは推定μ・Σを既知と置いた条件付きの結果です。最小分散は将来の最小リスクを
保証しません。テスト成績では等金額戦略と比較します。単一銘柄ベンチマークは売買コスト控除前です。
取引は判断の翌終値、最初の評価日は新ポジションの収益がなくコストだけ発生します。

## 限界・改善
共分散定常性・期待収益の推定精度を仮定します。実務の252営業日は近似です。
銘柄別上限、縮小共分散、ロバスト推定、ウォークフォワード、期間別検証を追加しましょう。
テストを見てモデルを調整したら、独立の最終評価期間を別途用意してください。